# pixabay-image-seed.ipynb — Semeadura da image-stock via léxico

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Varre `eventos-biblicos.js` (livro inteiro, ou livro + faixa de capítulo)
e busca imagens no Pixabay pra cada evento que **ainda não tem cobertura**
-- escreve direto na `pixabay-image-stock`, já com `Tags_Biblia_PT`
preenchida e uma referência de qual evento gerou cada linha
(`Chave_Match_ID`).

**Fluxo:**
1. Rode a célula de SEMEAR — adiciona linhas novas na planilha
2. Abra a planilha, revise, **apague as imagens que não servem**
3. Volte aqui e rode a célula de ALOCAR — o que sobrou vira entrada
   permanente na biblioteca de match (`biblioteca_match`), pronta pra
   qualquer vídeo futuro usar direto, sem gastar IA nem léxico de novo

Pra rodar a Bíblia inteira de uma vez (sem escolher livro/capítulo), use
`pixabay-image-seed-biblia-completa.ipynb` em vez deste — mesma lógica,
mas com delay mais longo entre buscas, pensado pra deixar rodando
sozinho por mais tempo.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread "mistralai>=1.2.0"

import shutil, sys, json
from pathlib import Path

from google.colab import drive, auth, userdata
from google.auth import default
import gspread

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

from groq import Groq
from mistralai.client import Mistral

GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None
MODELO_GROQ = "qwen/qwen3.6-27b"
MODELO_MISTRAL = "mistral-small-latest"

CHAVE_API_PIXABAY = userdata.get("PIXABAY_KEY")

print("✅ Setup concluído")
print(f"   Pixabay: {'disponível' if CHAVE_API_PIXABAY else '❌ PIXABAY_KEY não encontrada nos Secrets do Colab'}")
print(f"   Groq:    {'disponível' if groq_client else 'não configurado (GROQ_KEY ausente)'}")
print(f"   Mistral: {'disponível' if mistral_client else 'não configurado (MISTRAL_KEY ausente)'}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

PASTA_DRIVE_RAIZ = "narrated_video"

# ── Escopo da busca ──────────────────────────────────────────────────────
# Nome do livro EXATAMENTE como aparece em eventos-biblicos.js (ex:
# "Mateus", "1 Coríntios", "Apocalipse").
LIVRO_PT = "Mateus"

# Deixe os dois em None pra pegar o LIVRO INTEIRO. Preencha os dois pra
# restringir a uma faixa de capítulo (ex: só o capítulo que você está
# trabalhando agora no vídeo).
CAPITULO_INICIO = None
CAPITULO_FIM = None

# ── Planilha de imagens (mesma da image-stock usada no match/vídeo) ───────
ID_PLANILHA_IMAGENS = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"
NOME_ABA_IMAGENS = "image-stock"

# ── Biblioteca de match (planilha independente) ───────────────────────────
# Deixe em branco na primeira vez -- o notebook cria a planilha sozinho e
# IMPRIME o ID; copie pra cá depois pra reusar (senão cria uma nova toda vez).
ID_PLANILHA_BIBLIOTECA_MATCH = "1i67VxksAkWYx1cZ_QeoesGXsW28hcA0p5IIfhjx8VHE"
NOME_ABA_BIBLIOTECA_MATCH = "biblioteca_match"

# ── Léxico (mesmos arquivos usados no match-scene-verse.ipynb) ────────────
NOME_ARQUIVO_EVENTOS = "eventos-biblicos.json"
NOME_ARQUIVO_TITULOS = "titulos-biblicos.json"
PASTA_DADOS_LEXICO = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/dados_lexico"

# ── Busca ───────────────────────────────────────────────────────────────
QUANTIDADE_POR_TAG = 3   # imagens por TAG, no nivel EVENTO e TITULO (rodam poucas vezes por capitulo -- pode ser generoso)
QUANTIDADE_POR_TAG_VERSICULO = 1   # imagens por TAG, no nivel VERSICULO -- roda MUITAS vezes (1 por verso), por isso mais enxuto
MAX_TAGS_POR_ITEM = 15   # corta a lista de tags compiladas nesse tamanho (evita item com dezenas de tags virar dezenas de buscas)
USAR_TAGS_SEMELHANTES = True   # inclui tags_semelhantes na busca alem das tags base (mais diversidade, mais busca)
MIN_CANDIDATOS_PARA_PULAR = 5   # versiculo com esse tanto de candidato (ou mais) na image-stock nao busca de novo
DELAY_SEGUNDOS = 2   # espera entre uma busca e outra -- Pixabay limita a 100 req/60s

print("=" * 60)
print("⚙️  CONFIGURAÇÃO")
print("=" * 60)
faixa = f"cap {CAPITULO_INICIO}–{CAPITULO_FIM}" if CAPITULO_INICIO else "livro inteiro"
print(f"   Escopo:          {LIVRO_PT} ({faixa})")
print(f"   Imagens/tag (evento/titulo): {QUANTIDADE_POR_TAG}")
print(f"   Imagens/tag (versiculo):     {QUANTIDADE_POR_TAG_VERSICULO}")
print(f"   Max tags/item:               {MAX_TAGS_POR_ITEM}")
print(f"   Tags semelhantes:            {'ON' if USAR_TAGS_SEMELHANTES else 'off'}")
print(f"   Pula versiculo com >= {MIN_CANDIDATOS_PARA_PULAR} candidato(s) ja na image-stock")
print(f"   Delay:           {DELAY_SEGUNDOS}s")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR — carrega o léxico + abre as planilhas            ║
# ╚══════════════════════════════════════════════════════════════════╝
from drive_utils import DriveClient
from match_pipeline import (
    carregar_lexico_biblico, abrir_ou_criar_biblioteca_match, garantir_aba_versiculo_tags,
    carregar_biblioteca_match, versiculos_para_semear,
)
from pixabay_seed_pipeline import (
    garantir_aba_eventos_semeados, eventos_para_semear, carregar_eventos_semeados, carregar_itens_semeados,
    semear_por_evento, semear_por_titulo, semear_por_versiculo, carregar_tags_image_stock,
)

_drive = DriveClient.get()

# ── léxico (eventos + titulos -- titulos so e usado na passada por titulo/versiculo) ──
_dest_eventos = Path(NOME_ARQUIVO_EVENTOS)
_drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_EVENTOS, _dest_eventos)
if not _dest_eventos.exists():
    raise FileNotFoundError(f"{NOME_ARQUIVO_EVENTOS} não encontrado em {PASTA_DADOS_LEXICO} (local nem Drive).")
with open(_dest_eventos, encoding="utf-8") as _f:
    eventos_biblicos = json.load(_f)

_dest_titulos = Path(NOME_ARQUIVO_TITULOS)
_drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_TITULOS, _dest_titulos)
with open(_dest_titulos, encoding="utf-8") as _f:
    titulos_biblicos = json.load(_f)

print(f"✅ Léxico: {len(eventos_biblicos)} eventos, {len(titulos_biblicos)} títulos")

# ── planilhas ───────────────────────────────────────────────────────────
_spreadsheet_imagens = gc.open_by_key(ID_PLANILHA_IMAGENS)
sheet_imagens = _spreadsheet_imagens.worksheet(NOME_ABA_IMAGENS)

_spreadsheet_biblioteca, aba_biblioteca_match, _id_biblioteca_usado = abrir_ou_criar_biblioteca_match(gc, ID_PLANILHA_BIBLIOTECA_MATCH, NOME_ABA_BIBLIOTECA_MATCH)
aba_eventos_semeados = garantir_aba_eventos_semeados(_spreadsheet_biblioteca)
aba_versiculo_tags = garantir_aba_versiculo_tags(_spreadsheet_biblioteca)

print(f"✅ image-stock aberta ({len(sheet_imagens.get_all_records())} linha(s) hoje)")
print(f"✅ biblioteca_match aberta/criada")
print(f"✅ eventos_semeados aberta/criada")
print(f"✅ versiculo_tags aberta ({len(aba_versiculo_tags.get_all_records())} versículo(s) já taggeado(s))")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  SEMEAR POR TÍTULO — mais fino que evento (granularidade de   ║
# ║       versículo), roda 1x por título, não repete o que evento já   ║
# ║       buscou (cada nível só usa as PRÓPRIAS tags)                 ║
# ╚══════════════════════════════════════════════════════════════════╝
_titulos_ja_semeados = carregar_itens_semeados(aba_eventos_semeados, "titulo")  # mesma aba dos eventos, nivel "titulo" separado por dentro

_titulos_alvo = [
    (chave, entrada) for chave, entrada in titulos_biblicos.items()
    if entrada.get("livro") == LIVRO_PT
    and (CAPITULO_INICIO is None or (CAPITULO_INICIO <= entrada.get("capitulo", 0) <= (CAPITULO_FIM or CAPITULO_INICIO)))
    and chave not in _titulos_ja_semeados
]

print(f"📋 {len(_titulos_alvo)} título(s) sem cobertura ainda")
if _titulos_alvo:
    _cache_traducao, resumo_titulo = semear_por_titulo(
        _titulos_alvo, sheet_imagens, CHAVE_API_PIXABAY, cache_traducao={},
        aba_itens_semeados=aba_eventos_semeados,
        quantidade_por_tag=QUANTIDADE_POR_TAG, max_tags_por_item=MAX_TAGS_POR_ITEM,
        delay_segundos=DELAY_SEGUNDOS,
        groq_client=groq_client, mistral_client=mistral_client,
        modelo_groq=MODELO_GROQ, modelo_mistral=MODELO_MISTRAL,
        usar_tags_semelhantes=USAR_TAGS_SEMELHANTES,
    )
    print(f"\n✅ {sum(resumo_titulo.values())} imagem(ns) nova(s) adicionada(s) na image-stock")
else:
    print("Nada pra semear -- todos os títulos desse escopo já têm cobertura.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3️⃣  SEMEAR POR VERSÍCULO — a mais fina, roda 1x por versículo.   ║
# ║       Pula sozinho quem já tem vencedor OU já tem                  ║
# ║       MIN_CANDIDATOS_PARA_PULAR candidato(s) -- pra rodar só nos   ║
# ║       versículos que sobraram mal servidos pelas passadas acima.   ║
# ║                                                                    ║
# ║       Requer versiculo_tags preenchida (rode match-scene-verse     ║
# ║       .ipynb pra esse capítulo ANTES, senão não tem tag pra buscar)║
# ╚══════════════════════════════════════════════════════════════════╝
_versiculos_alvo_todos = versiculos_para_semear(aba_versiculo_tags, LIVRO_PT, CAPITULO_INICIO, CAPITULO_FIM)

if not _versiculos_alvo_todos:
    print(f"⚠️  Nenhum versículo com tag ainda pra {LIVRO_PT}"
          + (f" cap {CAPITULO_INICIO}" if CAPITULO_INICIO else "") +
          " -- rode match-scene-verse.ipynb pra esse capítulo primeiro.")
else:
    _biblioteca_match_dict = carregar_biblioteca_match(aba_biblioteca_match)
    _tags_image_stock = carregar_tags_image_stock(sheet_imagens)

    print(f"📋 {len(_versiculos_alvo_todos)} versículo(s) no escopo -- conferindo quem já está bem servido...")
    _cache_traducao, resumo_versiculo = semear_por_versiculo(
        _versiculos_alvo_todos, sheet_imagens, CHAVE_API_PIXABAY, cache_traducao={},
        aba_itens_semeados=aba_eventos_semeados,
        quantidade_por_tag=QUANTIDADE_POR_TAG_VERSICULO, max_tags_por_item=MAX_TAGS_POR_ITEM,
        delay_segundos=DELAY_SEGUNDOS,
        groq_client=groq_client, mistral_client=mistral_client,
        modelo_groq=MODELO_GROQ, modelo_mistral=MODELO_MISTRAL,
        usar_tags_semelhantes=USAR_TAGS_SEMELHANTES,
        tipo_fonte="imagem", biblioteca_match=_biblioteca_match_dict,
        tags_image_stock=_tags_image_stock, min_candidatos_para_pular=MIN_CANDIDATOS_PARA_PULAR,
    )
    print(f"\n✅ {sum(resumo_versiculo.values())} imagem(ns) nova(s) adicionada(s) na image-stock")

print("\n👉 Abra a planilha biblioteca_match e use o menu '📖 Revisão por Versículo' →")
print("   'Abrir painel' pra escolher o vencedor de cada versículo.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1️⃣  SEMEAR — busca no Pixabay e escreve na image-stock          ║
# ╚══════════════════════════════════════════════════════════════════╝
if not CHAVE_API_PIXABAY:
    raise RuntimeError("PIXABAY_KEY não encontrada -- confira os Secrets do Colab (ver célula de Setup).")

_ja_cobertos = carregar_eventos_semeados(aba_eventos_semeados)  # ja teve busca feita antes (nao depende da image-stock)
_alvo = eventos_para_semear(eventos_biblicos, LIVRO_PT, CAPITULO_INICIO, CAPITULO_FIM, ja_cobertos=_ja_cobertos)

print(f"📋 {len(_alvo)} evento(s) sem cobertura ainda (de {len(_ja_cobertos)} já semeados antes)")
if _alvo:
    _cache_traducao, resumo = semear_por_evento(
        _alvo, sheet_imagens, CHAVE_API_PIXABAY, cache_traducao={},
        aba_itens_semeados=aba_eventos_semeados,
        quantidade_por_tag=QUANTIDADE_POR_TAG, max_tags_por_item=MAX_TAGS_POR_ITEM,
        delay_segundos=DELAY_SEGUNDOS,
        groq_client=groq_client, mistral_client=mistral_client,
        modelo_groq=MODELO_GROQ, modelo_mistral=MODELO_MISTRAL,
        usar_tags_semelhantes=USAR_TAGS_SEMELHANTES,
    )
    print(f"\n✅ {sum(resumo.values())} imagem(ns) nova(s) adicionada(s) na image-stock")
else:
    print("Nada pra semear -- todo o escopo já tem cobertura.")

print("\n👉 Abra a planilha biblioteca_match e use o menu '📖 Revisão por Versículo' →")
print("   'Abrir painel' pra escolher o vencedor de cada versículo (compara texto + candidatos).")
print("\n⚠️  Essa é a passada por EVENTO -- por título e por versículo ainda são só do módulo")
print("   (semear_por_titulo/semear_por_versiculo), não estão ligadas neste notebook ainda.")


---
## 2️⃣ Depois de revisar a planilha manualmente, rode a célula abaixo:

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  ESCOLHER O VENCEDOR (agora é por versículo, não em lote)     ║
# ╚══════════════════════════════════════════════════════════════════╝
# A alocação automática (1 imagem por evento) foi substituída pelo
# painel de revisão -- cada versículo pode ter sua própria imagem,
# mesmo dentro do mesmo evento/título. Depois de rodar a célula 1
# acima (e revisar/apagar candidatos ruins na image-stock direto),
# abra a planilha biblioteca_match no Sheets e use o menu
# '📖 Revisão por Versículo' → 'Abrir painel' para escolher o vencedor
# de cada versículo, olhando o texto (biblia_texto) ao lado dos
# candidatos (image-stock) -- grava direto na biblioteca_match quando
# você clica.
